In [229]:
import tripodpy
from dustpy import hdf5writer
import numpy as np
wrtr = hdf5writer()

In [230]:
wrtr.datadir = "data_post_pipeline/prueba_sigma_componentes"
#wrtr.datadir = "data_post_pipeline/pipeline_v2"
data = wrtr.read.all()

In [231]:
data.grid.SigmaIce_H2O

array([[0.00000000e+00, 2.64308461e+00, 1.40325801e+00, 7.09907419e-01,
        3.29750124e-01, 1.31696557e-01, 4.02654585e-02, 7.67430495e-03,
        5.96026490e-12, 5.96026490e-12],
       [0.00000000e+00, 2.64308461e+00, 1.40325801e+00, 7.09907419e-01,
        3.29750124e-01, 1.31696557e-01, 4.02654585e-02, 7.67430495e-03,
        5.96026490e-12, 5.96026490e-12],
       [0.00000000e+00, 2.64308461e+00, 1.40325801e+00, 7.09907419e-01,
        3.29750124e-01, 1.31696557e-01, 4.02654585e-02, 7.67430495e-03,
        5.96026490e-12, 5.96026490e-12],
       [0.00000000e+00, 2.64308461e+00, 1.40325801e+00, 7.09907419e-01,
        3.29750124e-01, 1.31696557e-01, 4.02654585e-02, 7.67430495e-03,
        5.96026490e-12, 5.96026490e-12],
       [0.00000000e+00, 2.64308461e+00, 1.40325801e+00, 7.09907419e-01,
        3.29750124e-01, 1.31696557e-01, 4.02654585e-02, 7.67430495e-03,
        5.96026490e-12, 5.96026490e-12],
       [0.00000000e+00, 2.64308461e+00, 1.40325801e+00, 7.09907419e-01,
   

In [232]:
from pipeline_snowlines import WaterworldPipeline
import dustpy.constants as c

pipeline = WaterworldPipeline("data_post_pipeline/prueba_sigma_componentes")
pipeline.active_species = ["H2O", "CO2", "CO"]
pipeline.setup_grid(rmin=1*c.au, rmax=300*c.au, Nr=10)
pipeline.setup_star()
pipeline.initialize_simulation()
pipeline.add_volatile_components()
pipeline.setup_physics()
pipeline.setup_star_evolution()    # radio estelar contrae -> snowlines migran
pipeline.add_snowline_fields()     # rsnow_H2O/CO2/CO en HDF5
pipeline.add_ice_sigma_fields()    # SigmaIce_H2O/CO2/CO/silicates en HDF5
pipeline.sim.update()              # Por seguridad

print("\nPipeline completado.")

Configurando Grilla: 10 celdas entre 1.0 y 300.0 AU
Inicializando la simulación base...
Agregando sub-componentes (volátiles y refractarios) desde chem.txt...
  → Especies activas: ['H2O', 'CO2', 'CO']
Configurando física dinámica del hielo (v_frag updater)...
Configurando evolucion estelar (contraccion del radio)...
Agregando fields de posicion de snowline al grid...
  -> rsnow_H2O (Tsub=150.0K): 1.77 AU
  -> rsnow_CO2 (Tsub=70.0K): 9.79 AU
  -> rsnow_CO (Tsub=25.0K): 95.87 AU
Agregando fields de densidad de hielo por especie (SigmaIce_X)...
  -> SigmaIce_H2O (Tsub=150.0K, f=0.2980): max=2.643e+00 g/cm^2
  -> SigmaIce_CO2 (Tsub=70.0K, f=0.0132): max=1.466e-02 g/cm^2
  -> SigmaIce_CO (Tsub=25.0K, f=0.0265): max=5.298e-13 g/cm^2
  -> SigmaIce_silicates (Tsub=1500.0K, f=0.6623): max=1.059e+01 g/cm^2

Pipeline completado.


In [233]:
pipeline.run_integration(t_end_years=5, num_snapshots=10)

Configurando integración temporal hasta: 5.0e+00 años con 10 snapshots.
Empezando ejecución...

tripodpy v1.0.0

Writing file data_post_pipeline\prueba_sigma_componentes\data0000.hdf5
Writing dump file data_post_pipeline\prueba_sigma_componentes\frame.dmp
Writing file data_post_pipeline\prueba_sigma_componentes\data0001.hdf5
Writing dump file data_post_pipeline\prueba_sigma_componentes\frame.dmp
Writing file data_post_pipeline\prueba_sigma_componentes\data0002.hdf5
Writing dump file data_post_pipeline\prueba_sigma_componentes\frame.dmp
Writing file data_post_pipeline\prueba_sigma_componentes\data0003.hdf5
Writing dump file data_post_pipeline\prueba_sigma_componentes\frame.dmp
Writing file data_post_pipeline\prueba_sigma_componentes\data0004.hdf5
Writing dump file data_post_pipeline\prueba_sigma_componentes\frame.dmp
Writing file data_post_pipeline\prueba_sigma_componentes\data0005.hdf5
Writing dump file data_post_pipeline\prueba_sigma_componentes\frame.dmp
Writing file data_post_pipeli

In [234]:
pipeline.sim.components.H2O.dust.Sigma.sum(-1)

[4.48324521e+00 2.23886733e+00 4.47262391e-01 1.19758653e-01
 7.11377079e-02 5.75257265e-02 1.72134472e-02 3.77092954e-03
 4.07195505e-05 2.00000000e-11]

In [235]:
data.grid.SigmaIce_H2O[-1,]

array([0.00000000e+00, 2.64308461e+00, 1.40325801e+00, 7.09907419e-01,
       3.29750124e-01, 1.31696557e-01, 4.02654585e-02, 7.67430495e-03,
       5.96026490e-12, 5.96026490e-12])

In [236]:
help(pipeline.sim.gas.boundary.inner.setcondition)


Help on method setcondition in module dustpy.utils.boundary:

setcondition(condition, value=None) method of dustpy.utils.boundary.Boundary instance
    Function to set boundary condition.
    
    Parameters
    ----------
    condition : string
        Type of boundary conditon:
            - "const_grad" : constant gradient
            - "const_pow" : constant power law
            - "const_val" : constant value
            - "val" : custom value
            - "grad" : custom gradient
            - "pow" : custom power law with set exponent
            - None : Don't impose boundary condition (default)
    value : float or array, optional, default : None
        Value if needed for boundary condition



## Agregar Sigma para componentes ya que no se guarda, de hecho no exist ecomo field en las componentes


In [237]:
pipeline = WaterworldPipeline("data_post_pipeline/prueba_sigma_comp_v2")
pipeline.active_species = ["H2O", "CO2", "CO"]
pipeline.setup_grid(rmin=1*c.au, rmax=300*c.au, Nr=10)
pipeline.setup_star()
pipeline.initialize_simulation()
pipeline.add_volatile_components()
pipeline.setup_physics()
pipeline.setup_star_evolution()    # radio estelar contrae -> snowlines migran
pipeline.add_snowline_fields()     # rsnow_H2O/CO2/CO en HDF5
pipeline.add_ice_sigma_fields()    # SigmaIce_H2O/CO2/CO/silicates en HDF5
pipeline.sim.update()              # Por seguridad


Configurando Grilla: 10 celdas entre 1.0 y 300.0 AU
Inicializando la simulación base...
Agregando sub-componentes (volátiles y refractarios) desde chem.txt...
  → Especies activas: ['H2O', 'CO2', 'CO']
Configurando física dinámica del hielo (v_frag updater)...
Configurando evolucion estelar (contraccion del radio)...
Agregando fields de posicion de snowline al grid...
  -> rsnow_H2O (Tsub=150.0K): 1.77 AU
  -> rsnow_CO2 (Tsub=70.0K): 9.79 AU
  -> rsnow_CO (Tsub=25.0K): 95.87 AU
Agregando fields de densidad de hielo por especie (SigmaIce_X)...
  -> SigmaIce_H2O (Tsub=150.0K, f=0.2980): max=2.643e+00 g/cm^2
  -> SigmaIce_CO2 (Tsub=70.0K, f=0.0132): max=1.466e-02 g/cm^2
  -> SigmaIce_CO (Tsub=25.0K, f=0.0265): max=5.298e-13 g/cm^2
  -> SigmaIce_silicates (Tsub=1500.0K, f=0.6623): max=1.059e+01 g/cm^2


Ahora lo que trataremos de hacer es agregar los Sigma de los compomoentes tango el de polvo y hielo. De momento solo estas propiedades se guardan en el file

In [238]:
pipeline.sim.components.CO2.dust.Sigma.save = True
pipeline.sim.components.H2O.dust.Sigma.save = True
pipeline.sim.components.CO.dust.Sigma.save = True

In [239]:
pipeline.sim.components.H2O.gas.Sigma.save = True
pipeline.sim.components.CO2.gas.Sigma.save = True
pipeline.sim.components.CO.gas.Sigma.save = True


In [240]:
pipeline.run_integration(t_end_years=5, num_snapshots=10)

Configurando integración temporal hasta: 5.0e+00 años con 10 snapshots.
Empezando ejecución...

tripodpy v1.0.0

Writing file data_post_pipeline\prueba_sigma_comp_v2\data0000.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v2\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v2\data0001.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v2\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v2\data0002.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v2\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v2\data0003.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v2\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v2\data0004.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v2\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v2\data0005.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v2\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v2\data0006.hdf5
Writing du

In [241]:
import tripodpy
from dustpy import hdf5writer
import numpy as np
wrtr = hdf5writer()
wrtr.datadir = "data_post_pipeline/prueba_sigma_comp_v2"
#wrtr.datadir = "data_post_pipeline/pipeline_v2"
data = wrtr.read.all()

In [242]:
data.components.CO.dust

namespace(Fi=array([[[-4.38074991e-11, -4.82268908e-10],
                     [-6.07361815e-11, -4.99197128e-10],
                     [-7.62651809e-11, -6.00076273e-10],
                     [-7.03153267e-11, -1.01166816e-10],
                     [-1.83221267e-09, -1.32944062e-09],
                     [-9.92320529e-08, -7.54400951e-08],
                     [-2.95702528e-05, -2.27960007e-05],
                     [-4.73438098e-02, -3.86162307e-02],
                     [ 4.06031269e-02,  2.13976841e-02],
                     [-3.36855389e-05, -8.69761106e-06],
                     [ 0.00000000e+00,  0.00000000e+00]],
             
                    [[-4.36906262e-11, -3.72258049e-10],
                     [-6.06169027e-11, -3.89184005e-10],
                     [-7.68576515e-11, -7.52827547e-10],
                     [-7.18036840e-11, -5.22798223e-10],
                     [-1.41814569e-09, -6.65296483e-11],
                     [-9.22500947e-08, -6.95412136e-08],
                

### No funciono con el Sigma.save = True, intentemos agregar el valor con el diastole de dust.Sigma y gas.Sigma de cada componente.

### Mejor activare el tracer = True para el gas y polvo en la parte de las componentes si es posible hacerlo, quizas asi se guarda el Sigma para ambos casos

In [243]:
pipeline = WaterworldPipeline("data_post_pipeline/prueba_sigma_comp_v3")
pipeline.active_species = ["H2O", "CO2", "CO"]
pipeline.setup_grid(rmin=1*c.au, rmax=300*c.au, Nr=10)
pipeline.setup_star()
pipeline.initialize_simulation()
pipeline.add_volatile_components()
pipeline.setup_physics()
pipeline.setup_star_evolution()    # radio estelar contrae -> snowlines migran
pipeline.add_snowline_fields()     # rsnow_H2O/CO2/CO en HDF5
pipeline.add_ice_sigma_fields()    # SigmaIce_H2O/CO2/CO/silicates en HDF5
pipeline.sim.update()              # Por seguridad


Configurando Grilla: 10 celdas entre 1.0 y 300.0 AU
Inicializando la simulación base...
Agregando sub-componentes (volátiles y refractarios) desde chem.txt...
  → Especies activas: ['H2O', 'CO2', 'CO']
Configurando física dinámica del hielo (v_frag updater)...
Configurando evolucion estelar (contraccion del radio)...
Agregando fields de posicion de snowline al grid...
  -> rsnow_H2O (Tsub=150.0K): 1.77 AU
  -> rsnow_CO2 (Tsub=70.0K): 9.79 AU
  -> rsnow_CO (Tsub=25.0K): 95.87 AU
Agregando fields de densidad de hielo por especie (SigmaIce_X)...
  -> SigmaIce_H2O (Tsub=150.0K, f=0.2980): max=2.643e+00 g/cm^2
  -> SigmaIce_CO2 (Tsub=70.0K, f=0.0132): max=1.466e-02 g/cm^2
  -> SigmaIce_CO (Tsub=25.0K, f=0.0265): max=5.298e-13 g/cm^2
  -> SigmaIce_silicates (Tsub=1500.0K, f=0.6623): max=1.059e+01 g/cm^2


In [244]:
p = pipeline

In [245]:
p.sim.components.updateorder

['Default', np.str_('H2O'), np.str_('CO2'), np.str_('CO'), 'silicates']

In [246]:
p.sim.components.silicates.updater.diastole

Updater
-------

Type: NoneType

Notamos que no existe un diastole (accion luego de caclular los silicates)

In [247]:
p.run_integration(t_end_years=5, num_snapshots=10)

Configurando integración temporal hasta: 5.0e+00 años con 10 snapshots.
Empezando ejecución...

tripodpy v1.0.0

Writing file data_post_pipeline\prueba_sigma_comp_v3\data0000.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v3\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v3\data0001.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v3\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v3\data0002.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v3\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v3\data0003.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v3\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v3\data0004.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v3\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v3\data0005.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v3\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v3\data0006.hdf5
Writing du

In [248]:
import tripodpy
from dustpy import hdf5writer
import numpy as np
wrtr = hdf5writer()
wrtr.datadir = "data_post_pipeline/prueba_sigma_comp_v3"
#wrtr.datadir = "data_post_pipeline/pipeline_v2"
data = wrtr.read.all()

### No funciono el tracer para ningun caso, pero ocuparamos eso que ya dijismos, vamos a ocupar np.array para agregar los datos cada vez que se actualicen.

In [249]:
pipeline = WaterworldPipeline("data_post_pipeline/prueba_sigma_comp_v3")
pipeline.active_species = ["H2O", "CO2", "CO"]
pipeline.setup_grid(rmin=1*c.au, rmax=300*c.au, Nr=10)
pipeline.setup_star()
pipeline.initialize_simulation()
pipeline.add_volatile_components()
pipeline.setup_physics()
pipeline.setup_star_evolution()    # radio estelar contrae -> snowlines migran
pipeline.add_snowline_fields()     # rsnow_H2O/CO2/CO en HDF5
pipeline.add_ice_sigma_fields()    # SigmaIce_H2O/CO2/CO/silicates en HDF5
pipeline.sim.update()              # Por seguridad
p = pipeline

Configurando Grilla: 10 celdas entre 1.0 y 300.0 AU
Inicializando la simulación base...
Agregando sub-componentes (volátiles y refractarios) desde chem.txt...
  → Especies activas: ['H2O', 'CO2', 'CO']
Configurando física dinámica del hielo (v_frag updater)...
Configurando evolucion estelar (contraccion del radio)...
Agregando fields de posicion de snowline al grid...
  -> rsnow_H2O (Tsub=150.0K): 1.77 AU
  -> rsnow_CO2 (Tsub=70.0K): 9.79 AU
  -> rsnow_CO (Tsub=25.0K): 95.87 AU
Agregando fields de densidad de hielo por especie (SigmaIce_X)...
  -> SigmaIce_H2O (Tsub=150.0K, f=0.2980): max=2.643e+00 g/cm^2
  -> SigmaIce_CO2 (Tsub=70.0K, f=0.0132): max=1.466e-02 g/cm^2
  -> SigmaIce_CO (Tsub=25.0K, f=0.0265): max=5.298e-13 g/cm^2
  -> SigmaIce_silicates (Tsub=1500.0K, f=0.6623): max=1.059e+01 g/cm^2


In [251]:
import numpy as np

np.zeros(10)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

### Ahora corregimos el codigo y vemos si sirve.

In [1]:
from pipeline_snowlines import WaterworldPipeline
import dustpy.constants as c
pipeline = WaterworldPipeline("data_post_pipeline/prueba_sigma_comp_v4")
pipeline.active_species = ["H2O", "CO2", "CO"]
pipeline.setup_grid(rmin=1*c.au, rmax=300*c.au, Nr=10)
pipeline.setup_star()
pipeline.initialize_simulation()
pipeline.add_volatile_components()
pipeline.setup_physics()
pipeline.setup_star_evolution()    # radio estelar contrae -> snowlines migran
pipeline.add_snowline_fields()     # rsnow_H2O/CO2/CO en HDF5
pipeline.add_ice_sigma_fields()    # SigmaIce_H2O/CO2/CO/silicates en HDF5
pipeline.sim.update()              # Por seguridad
p = pipeline

Configurando Grilla: 10 celdas entre 1.0 y 300.0 AU
Inicializando la simulación base...
Agregando sub-componentes (volátiles y refractarios) desde chem.txt...
  → Especies activas: ['H2O', 'CO2', 'CO']
Configurando física dinámica del hielo (v_frag updater)...
  → H2O: T < 150 K → v_frag = 1000 cm/s (10 m/s)
  → CO2: T < 70 K → v_frag = 500 cm/s (5 m/s)
  → CO: T < 25 K → v_frag = 300 cm/s (3 m/s)
Configurando evolucion estelar (contraccion del radio)...
Agregando fields de posicion de snowline al grid...
  -> rsnow_H2O (Tsub=150.0K): 1.77 AU
  -> rsnow_CO2 (Tsub=70.0K): 9.79 AU
  -> rsnow_CO (Tsub=25.0K): 95.87 AU
Agregando fields de densidad superficial por componente (SigmaDust/SigmaGas)...
  → SigmaDust_H2O: max = 2.000e-10 g/cm²
  → SigmaGas_H2O:  max = 3.238e+00 g/cm²
  → SigmaDust_CO2: max = 2.000e-10 g/cm²
  → SigmaGas_CO2:  max = 2.226e+00 g/cm²
  → SigmaDust_CO: max = 2.000e-10 g/cm²
  → SigmaGas_CO:  max = 2.833e+00 g/cm²


In [2]:
p.sim.components

Group (components)
------------------
    CO           : Group (CO (dust_tracer=False, gas_active=True, gas_tracer=False))
    CO2          : Group (CO2 (dust_tracer=False, gas_active=True, gas_tracer=False))
    Default      : Group (Default gas componentDefault (dust_tracer=False, gas_active=True, gas_tracer=False))
    H2O          : Group (H2O (dust_tracer=False, gas_active=True, gas_tracer=False))
    silicates    : Group (silicates (dust_tracer=False, gas_active=False, gas_tracer=False))
  -----

In [3]:
p.sim.grid.SigmaGas_CO

[2.83299455 1.57097886 0.83405906 0.42195    0.19599466 0.07827691
 0.02393271 0.0045614  0.         0.        ]

In [4]:
from tripodpy import Simulation

In [5]:
sim = Simulation()

sim.initialize()

In [6]:
sim.gas.mu.append

AttributeError: 'Field' object has no attribute 'append'

In [2]:
p.run_integration(t_end_years=5, num_snapshots=10)

Configurando integración temporal hasta: 5.0e+00 años con 10 snapshots.
Empezando ejecución...

tripodpy v1.0.0

Writing file data_post_pipeline\prueba_sigma_comp_v4\data0000.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v4\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v4\data0001.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v4\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v4\data0002.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v4\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v4\data0003.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v4\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v4\data0004.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v4\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v4\data0005.hdf5
Writing dump file data_post_pipeline\prueba_sigma_comp_v4\frame.dmp
Writing file data_post_pipeline\prueba_sigma_comp_v4\data0006.hdf5
Writing du

In [3]:
import tripodpy
from dustpy import hdf5writer
import numpy as np
wrtr = hdf5writer()
wrtr.datadir = "data_post_pipeline/prueba_sigma_comp_v4"
#wrtr.datadir = "data_post_pipeline/pipeline_v2"
data = wrtr.read.all()

In [17]:
data.grid.SigmaGas_H2O[-1]


array([1.19073393e-003, 5.59008111e-004, 1.85435589e-009, 3.90466694e-015,
       2.94180135e-021, 3.26742810e-027, 1.24494607e-032, 1.89622442e-037,
       2.64985178e-037, 1.00000000e-100])

In [20]:
p.sim.components.H2O.gas.Sigma

[1.19073393e-003 5.59008111e-004 1.85435589e-009 3.90466694e-015
 2.94180135e-021 3.26742810e-027 1.24494607e-032 1.89622442e-037
 2.64985178e-037 1.00000000e-100]

---
Todo fue creado con exito, ahora el pipeline registra los sigma en cada snapshot.

In [1]:
from tripodpy import Simulation

In [2]:
sim = Simulation()

In [4]:
sim.dust

Group (Dust quantities)
-----------------------
    backreaction : Group (Backreaction coefficients)
    boundary     : Group (Boundary conditions)
    delta        : Group (Mixing parameters)
    f            : Group (Fudge factors)
    Fi           : Group (Fluxes)
    p            : Group (Probabilities)
    q            : Group (Distribution exponents)
    S            : Group (Sources)
    s            : Group (Characteristic particle sizes)
    v            : Group (Velocities)
  -----
    a            : NoneType
    D            : NoneType
    eps          : NoneType
    fill         : NoneType
    H            : NoneType
    m            : NoneType
    qrec         : NoneType
    rho          : NoneType
    rhos         : NoneType
    Sigma        : NoneType
    SigmaFloor   : NoneType
    St           : NoneType
  -----